In [2]:
from pathlib import Path
import geopandas as gpd
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"

PARCELS_DIR = DATA_DIR / "parcels" / "processed"

parcels_in_village_path = PARCELS_DIR / "parcels_assessor_village_no_duplicates.gpkg"

parcels_in_village_path

WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/parcels/processed/parcels_assessor_village_no_duplicates.gpkg')

In [3]:
parcels_in_village = gpd.read_file(parcels_in_village_path)

print(parcels_in_village.columns)
print(len(parcels_in_village))

Index(['APN', 'Floor', 'LandSize', 'StartDate', 'Shp_Area', 'Shp_Length',
       'PARCEL_NO', 'PROPERTYUSECODE', 'FULLCASHVALUE', 'LANDFULLCASHVALUE',
       'IMPRFULLCASHVALUE', 'LPVAMOUNT', 'LPVASSESSEDVALUE', 'SQFOOTAGE',
       'NUMBEROFUNITS', 'NEIGHBORHOODCODE', 'MARKETAREACODE', 'index_right',
       'NAME', 'geometry'],
      dtype='object')
1736192


In [4]:
print(parcels_in_village.crs)

EPSG:2868


In [5]:
parcels = parcels_in_village.copy()

# Area in acres from geometry (sq ft / 43,560)
parcels["area_acres"] = parcels.geometry.area / 43560.0

# Drop anything with zero/negative area just in case
parcels = parcels[parcels["area_acres"] > 0]

In [6]:
village_parcels = parcels[parcels["NAME"].notna()].copy()
print(len(village_parcels))

528085


In [7]:
group_cols = ["index_right", "NAME"]

village_stats = (
    village_parcels
    .groupby(group_cols, as_index=False)
    .agg(
        total_taxable_value=("LPVASSESSEDVALUE", "sum"),
        total_full_cash=("FULLCASHVALUE", "sum"),
        total_land_value=("LANDFULLCASHVALUE", "sum"),
        total_acres=("area_acres", "sum"),
        parcel_count=("APN", "count"),
    )
)

village_stats["taxable_value_per_acre"] = (
    village_stats["total_taxable_value"] / village_stats["total_acres"]
)

village_stats.head()

,index_right,NAME,total_taxable_value,total_full_cash,total_land_value,total_acres,parcel_count,taxable_value_per_acre
0,0.0,Ahwatukee Foothills,1.197546e+09,1.983603e+10,5.383996e+09,20959.303651,30686,57136.742324
1,1.0,Laveen,4.859999e+08,1.019906e+10,2.939275e+09,17295.368170,26951,28100.003840
2,2.0,South Mountain,1.115734e+09,2.112627e+10,5.686000e+09,22195.046699,42959,50269.498105
3,3.0,Estrella,1.481555e+09,2.224583e+10,5.216087e+09,23296.375678,30007,63595.934511
4,4.0,Central City,2.152483e+09,2.841179e+10,7.045698e+09,10688.098635,19001,201390.674568


In [9]:
out_csv_path = PROJECT_ROOT / "outputs" / "village_land_value_stats.csv"
village_stats.to_csv(out_csv_path, index=False)